# Assignment 3

### Part A & B - Setup

I am loading the same 100 high-risk examples as i used in assignment 2

In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [16]:
full_data = pd.read_parquet("patents_50k_green.parquet")
full_data.head()

,id,date,text,Y02A,Y02B,Y02C,Y02D,Y02E,Y02P,Y02T,Y02W,is_green_silver
0,8929789,2015-01-06,"1. A fixing device comprising: a rotatable, en...",0,0,0,0,0,0,0,0,0
1,9253996,2016-02-09,1. A method of recovering pectin from citrus p...,0,0,0,0,1,1,0,0,1
2,9580824,2017-02-28,1. An anion-conducting polymeric membrane comp...,0,0,0,0,1,0,0,0,1
3,9383068,2016-07-05,"1. An LED based lighting system, comprising: a...",0,1,0,0,0,0,0,0,1
4,8915542,2014-12-23,1. A sunroof apparatus comprising: a movable r...,0,0,0,0,0,0,0,0,0


In [17]:
#Loading the high-risk examples

df = pd.read_csv('hitl_green_100.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   doc_id   100 non-null    int64  
 1   text     100 non-null    str    
 2   p_green  100 non-null    float64
 3   u        100 non-null    float64
 4   label_1  0 non-null      float64
 5   label_2  0 non-null      float64
 6   label_3  0 non-null      float64
dtypes: float64(5), int64(1), str(1)
memory usage: 96.9 KB


In [18]:
df.head()

,doc_id,text,p_green,u,label_1,label_2,label_3
0,9447102,1. A compound of Formula 1: wherein R is selec...,0.499943,0.999886,NaN,NaN,NaN
1,8607927,1. A multilayer laminate for use as a flame ba...,0.499938,0.999876,NaN,NaN,NaN
2,8984809,1. Apparatus for converting a door operator wh...,0.499823,0.999647,NaN,NaN,NaN
3,8809354,1. A pharmaceutical composition comprising a c...,0.500248,0.999504,NaN,NaN,NaN
4,9227385,1. A method for processing a translucent rigid...,0.499463,0.998926,NaN,NaN,NaN


## Part C - Option 1

I will for part C choose to follow option 1 where i will create a multi-agent system workflow where i will create three different agents that will debate and label the 100 claims

#### Configuration of API

In [19]:
import os
import json
import time
from crewai import Agent, Task, Crew, LLM
from dotenv import load_dotenv

load_dotenv()  # reads the .env file


SLEEP_BETWEEN_CLAIMS = 60  # seconds


llm = LLM(
    model="groq/meta-llama/llama-4-scout-17b-16e-instruct",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.2,
)



In [20]:
# Setting up the Crew and Task
advocate = Agent(
    role="Green Patent Advocate",
    goal="Argue why this patent claim qualifies as Y02 green technology.",
    backstory="Expert in Y02 green technology patents.",
    llm=llm,
    verbose=False,
)

skeptic = Agent(
    role="Greenwashing Skeptic",
    goal="Challenge the Y02 classification and identify greenwashing.",
    backstory="Critical analyst who scrutinises environmental patent claims.",
    llm=llm,
    verbose=False,
)

judge = Agent(
    role="Patent Classification Judge",
    goal="Weigh both arguments and output a final JSON classification.",
    backstory="Neutral senior patent examiner.",
    llm=llm,
    verbose=False,
)

In [21]:
import os

# Skip already successful claims
if os.path.exists("classified_results.csv"):
    existing_df = pd.read_csv("classified_results.csv")
    done_ids = set(existing_df[existing_df["status"] == "success"]["doc_id"].tolist())
    results = [r for r in existing_df.to_dict("records") if r["status"] == "success"]
    print(f"Resuming — {len(done_ids)} already done, {len(df) - len(done_ids)} remaining")
else:
    done_ids = set()
    results = []

for i, row in df.iterrows():
    doc_id = row["doc_id"]

    if doc_id in done_ids:
        print(f"Skipping doc_id {doc_id} — already classified")
        continue

    claim_text = row["text"][:500]
    print(f"\n[{i+1}/{len(df)}] Classifying doc_id: {doc_id}...")

    advocate_task = Task(
        description=f"Argue why this patent claim qualifies as Y02 green technology:\n\n{claim_text}",
        agent=advocate,
        expected_output="A short argument (2-3 sentences) for why this claim is Y02 green technology."
    )
    skeptic_task = Task(
        description=f"Challenge the Y02 classification of this patent claim:\n\n{claim_text}",
        agent=skeptic,
        expected_output="A short counter-argument (2-3 sentences) against the Y02 classification."
    )
    judge_task = Task(
        description=(
            f"Classify this patent claim:\n\n{claim_text}\n\n"
            "Respond ONLY with valid JSON, no markdown:\n"
            '{"label": 1 or 0, "rationale": "2-3 sentence explanation"}'
        ),
        agent=judge,
        expected_output='{"label": 1, "rationale": "..."}',
        context=[advocate_task, skeptic_task]
    )

    crew = Crew(
        agents=[advocate, skeptic, judge],
        tasks=[advocate_task, skeptic_task, judge_task],
        verbose=False,
    )

    try:
        result = crew.kickoff()
        raw = str(result).strip()

        if "```" in raw:
            parts = raw.split("```")
            raw = parts[1] if len(parts) > 1 else parts[0]
            if raw.startswith("json"):
                raw = raw[4:]
        raw = raw.strip()

        parsed = json.loads(raw)
        parsed["doc_id"] = doc_id
        parsed["advocate_argument"] = str(crew.tasks[0].output)
        parsed["skeptic_argument"] = str(crew.tasks[1].output)
        parsed["status"] = "success"
        results.append(parsed)
        done_ids.add(doc_id)
        pd.DataFrame(results).to_csv("classified_results.csv", index=False)
        print(f"  → Label: {parsed.get('label')} | Status: success")

    except Exception as e:
        print(f"  ERROR — {str(e)}")
        print(f"  Stopping and saving {len(results)} successful results so far.")
        pd.DataFrame(results).to_csv("classified_results.csv", index=False)
        break  # stops the loop on first error

    if i + 1 < len(df):
        print(f"  Waiting {SLEEP_BETWEEN_CLAIMS}s...")
        time.sleep(SLEEP_BETWEEN_CLAIMS)

print("\nDone!")
print(f"Saved {len(results)} successful classifications to classified_results.csv")

Resuming — 100 already done, 0 remaining
Skipping doc_id 9447102 — already classified
Skipping doc_id 8607927 — already classified
Skipping doc_id 8984809 — already classified
Skipping doc_id 8809354 — already classified
Skipping doc_id 9227385 — already classified
Skipping doc_id 8862301 — already classified
Skipping doc_id 8527189 — already classified
Skipping doc_id 8383451 — already classified
Skipping doc_id 9469392 — already classified
Skipping doc_id 9842862 — already classified
Skipping doc_id 8994429 — already classified
Skipping doc_id 9306912 — already classified
Skipping doc_id 9723727 — already classified
Skipping doc_id 8597928 — already classified
Skipping doc_id 8613191 — already classified
Skipping doc_id 9711196 — already classified
Skipping doc_id 9244718 — already classified
Skipping doc_id 8625297 — already classified
Skipping doc_id 9809663 — already classified
Skipping doc_id 9366165 — already classified
Skipping doc_id 9446628 — already classified
Skipping doc_i

In [22]:
df_results = pd.read_csv("classified_results.csv")
df_results.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   label              100 non-null    float64
 1   rationale          100 non-null    str    
 2   doc_id             100 non-null    int64  
 3   advocate_argument  100 non-null    str    
 4   skeptic_argument   100 non-null    str    
 5   status             100 non-null    str    
dtypes: float64(1), int64(1), str(4)
memory usage: 213.1 KB


In [24]:
df_results['label']=df_results['label'].astype(int)
df_results['label'].value_counts()

label
0    69
1    31
Name: count, dtype: int64

In [25]:
# Add docid and text to the classified results for easier analysis
df_merged = df_results.merge(df[["doc_id", "text"]], on="doc_id", how="left").reset_index(drop=True)
df_merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   label              100 non-null    int64
 1   rationale          100 non-null    str  
 2   doc_id             100 non-null    int64
 3   advocate_argument  100 non-null    str  
 4   skeptic_argument   100 non-null    str  
 5   status             100 non-null    str  
 6   text               100 non-null    str  
dtypes: int64(2), str(5)
memory usage: 305.2 KB


#### Part D Human Review & Final Integration

In [ ]:
from datetime import datetime

def manual_hitl_review(df_results):
    """
    Human-in-the-loop review: manual review of ALL claims.
    df_results: your classified_results.csv merged with original text column
    """
    num_claims = len(df_results)
    df = df_results.copy()

    print("\n" + "="*80)
    print("HUMAN-IN-THE-LOOP REVIEW")
    print("="*80)
    print(f"Total claims to review manually: {num_claims}")
    print("="*80)

    for idx, row in df.iterrows():
        print(f"\n{'='*80}")
        print(f"CLAIM {idx+1}/{num_claims} — doc_id: {row['doc_id']}")
        print(f"{'='*80}")

        print(f"\nCLAIM TEXT:\n{row['text'][:400]}...")

        print(f"\nAI VERDICT:")
        verdict = "GREEN (1)" if row['label'] == 1 else "NOT GREEN (0)"
        print(f"   Suggestion: {verdict}")
        print(f"   Rationale: {row['rationale']}")

        print(f"\nYOUR TURN (Manual Review):")
        while True:
            human_label = input("   Is this GREEN? (1=yes, 0=no, Enter=agree with AI): ").strip()
            if human_label == "":
                human_label = int(row["label"])
                break
            elif human_label in ["0", "1"]:
                human_label = int(human_label)
                break
            else:
                print("   Please enter 0, 1 or press Enter to agree")

        notes = input("   Notes (optional, press Enter to skip): ").strip()
        if not notes:
            notes = "Manual - agreed" if human_label == row["label"] else "Manual - disagreed"

        if human_label == row["label"]:
            print("   ✓ You AGREED with the AI")
        else:
            print("   ✗ You DISAGREED with the AI")

        df.at[idx, "is_green_gold"] = human_label
        df.at[idx, "human_notes"] = notes
        df.at[idx, "reviewed_at"] = datetime.now().isoformat()

        # Progress tracker
        completed = idx + 1
        remaining = num_claims - completed
        print(f"\n   Progress: {completed}/{num_claims} complete — {remaining} remaining")

    print("\n" + "="*80)
    print("HITL REVIEW COMPLETE!")
    print("="*80)

    total = len(df)
    agreement = (df["label"] == df["is_green_gold"]).sum()
    agreement_rate = 100 * agreement / total

    print(f"\nSUMMARY:")
    print(f"Total claims reviewed: {total}")
    print(f"AI suggested GREEN: {int(df['label'].sum())}")
    print(f"Human labeled GREEN: {int(df['is_green_gold'].sum())}")
    print(f"Agreement rate: {agreement}/{total} ({agreement_rate:.1f}%)")

    disagreements = df[df["label"] != df["is_green_gold"]]
    print(f"Disagreements: {len(disagreements)}")
    if len(disagreements) > 0:
        print("\nDisagreed on doc_ids:", list(disagreements["doc_id"]))

    return df

# ── RUN ────────────────────────────────────────────────
hitl_final = manual_hitl_review(df_merged)


HUMAN-IN-THE-LOOP REVIEW
Total claims to review manually: 100

CLAIM 1/100 — doc_id: 8862301

CLAIM TEXT:
1. An inverted pendulum type vehicle including at least a first moving operation unit operable to move on said floor surface, a first actuator device for driving the first moving operation unit, a base assembled with the first moving operation unit and the first actuator device, and an occupant boarding member assembled to the base so as to be tiltable with respect to a vertical direction, the fir...

AI VERDICT:
   Suggestion: NOT GREEN (0)
   Rationale: The inverted pendulum type vehicle's primary purpose is to provide a novel mobility solution, with no explicit connection to environmental sustainability or reduced environmental impact. While the vehicle may potentially be used with environmentally friendly propulsion systems, the claim itself does not specify or imply any green technology features, making the Y02 classification seem like an example of greenwashing. The vehicle'

In [27]:
hitl_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   label              100 non-null    int64  
 1   rationale          100 non-null    str    
 2   doc_id             100 non-null    int64  
 3   advocate_argument  100 non-null    str    
 4   skeptic_argument   100 non-null    str    
 5   status             100 non-null    str    
 6   text               100 non-null    str    
 7   is_green_gold      100 non-null    float64
 8   human_notes        100 non-null    str    
 9   reviewed_at        100 non-null    str    
dtypes: float64(1), int64(2), str(7)
memory usage: 311.5 KB


In [29]:
hitl_final['is_green_gold']=hitl_final['is_green_gold'].astype(int)
hitl_final['is_green_gold'].value_counts()

is_green_gold
0    67
1    33
Name: count, dtype: int64

In [30]:
# Show disagreements
disagreements = hitl_final[hitl_final["label"] != hitl_final["is_green_gold"]]
if len(disagreements) > 0:
    print("\nDisagreements:")
    print(disagreements[["doc_id", "label", "is_green_gold", "human_notes"]].to_string())
else:
    print("\nNo disagreements!")


Disagreements:
     doc_id  label  is_green_gold         human_notes
3   9469392      0              1  Manual - disagreed
4   9842862      0              1  Manual - disagreed
7   8607927      1              0  Manual - disagreed
9   8809354      0              1  Manual - disagreed
13  8597928      1              0  Manual - disagreed
48  9701969      0              1  Manual - disagreed


In [31]:
#save the final results
hitl_final.to_csv("hitl_final.csv", index=False)
print("\nSaved to hitl_final.csv")


Saved to hitl_final.csv


### Fine-tune PatentSBERTa

Finetune with One LLM and HITL scores:
1️⃣ Evaluation on eval_silver (5,000 samples):
 [313/313 00:29]
  eval_loss: 0.4001
  eval_accuracy: 0.8172
  eval_precision: 0.8246
  eval_recall: 0.8115
  eval_f1: 0.8180
  eval_runtime: 29.1117
  eval_samples_per_second: 171.7520
  eval_steps_per_second: 10.7520
  epoch: 1.0000

2️⃣ Evaluation on gold_100 (HITL labeled):
  eval_loss: 0.7173
  eval_accuracy: 0.5400
  eval_precision: 0.9388
  eval_recall: 0.5169
  eval_f1: 0.6667
  eval_runtime: 0.6235
  eval_samples_per_second: 160.3900
  eval_steps_per_second: 11.2270
  epoch: 1.0000

Training set size: 40100
Eval set size: 5000
Gold set size: 100

  Results:
  Eval F1: 0.8180
  Gold F1: 0.6667

So after having finetuned the model and tested it on the eval_silver dataset we can compare the performance of the three models we have worked with so far. First we will load in the eval_metrics from the model in this assignment:

In [4]:
import pandas as pd
eval_df_3 = pd.read_csv("eval_metrics.csv")
eval_df_3

,test_loss,test_accuracy,test_f1,test_precision,test_recall,test_runtime,test_samples_per_second,test_steps_per_second
0,0.44341,0.82,0.824219,0.814986,0.833663,50.2545,99.494,3.124


We can recall the results from assignment 2 and compare the models: 

| Model Version | Training Data Source | F1 Score (Eval Set) |
|---------------|---------------------|---------------------|
| 1. Baseline | Frozen Embeddings (No Fine-tuning) | [0.78] |
| 2. Assignment 2 Model | Fine-tuned on Silver + Gold (Simple LLM) | [0.818] |
| 3. Assignment 3 Model | Fine-tuned on Silver + Gold (Agents - CrewAI) | [0.824] |